# Fongbe ASR Training - Google Colab

Fine-tuning Whisper for Fongbe speech recognition using LoRA.

## Prerequisites
1. Runtime → GPU (T4 recommended)
2. Upload `fongbe_dataset.tar.gz` to `MyDrive/fongbe/`

## 1. Setup Environment

In [2]:
from pathlib import Path
from google.colab import drive
import subprocess
import tarfile

# Mount Drive
drive.mount('/content/drive')

# Config paths
PROJECT_ROOT = Path('/content/drive/MyDrive/fongbe')
DATASET_TAR = PROJECT_ROOT / 'fongbe_dataset.tar.gz'
DATA_ROOT = PROJECT_ROOT / 'data' / 'processed' / 'fongbe_asr_unified'
OUTPUT_ROOT = PROJECT_ROOT / 'outputs'
REPO_URL = 'https://github.com/Appolinairee/fongbe-asr.git'

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"✓ Project: {PROJECT_ROOT}")

Mounted at /content/drive
✓ Project: /content/drive/MyDrive/fongbe


## 2. Clone Repository

In [9]:
import os

os.chdir('/content')
if not Path('fongbe-asr').exists():
    !git clone {REPO_URL} fongbe-asr
else:
    !cd fongbe-asr && git pull

os.chdir('fongbe-asr')
print(f"✓ Working dir: {Path.cwd()}")

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 434 bytes | 434.00 KiB/s, done.
From https://github.com/Appolinairee/fongbe-asr
   dff5133..ac365ea  main       -> origin/main
Updating dff5133..ac365ea
Fast-forward
 scripts/finetune_whisper.py | 4 ++--
 1 file changed, 2 insertions(+), 2 deletions(-)
✓ Working dir: /content/fongbe-asr


## 3. Extract Dataset

In [4]:
if not (DATA_ROOT / 'train').exists():
    if DATASET_TAR.exists():
        print(f"Extracting {DATASET_TAR.name}...")
        with tarfile.open(DATASET_TAR, 'r:gz') as tar:
            tar.extractall(PROJECT_ROOT / 'data' / 'processed')
        print("✓ Dataset extracted")
    else:
        raise FileNotFoundError(f"Dataset not found: {DATASET_TAR}")
else:
    print("✓ Dataset already extracted")

# Verify
n_train = len(list((DATA_ROOT / 'train').glob('*.arrow')))
print(f"✓ {n_train} files in train/")

Extracting fongbe_dataset.tar.gz...
✓ Dataset extracted
✓ 1 files in train/


/tmp/ipykernel_1999/3317955445.py:5: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(PROJECT_ROOT / 'data' / 'processed')


## 4. Install Dependencies

In [5]:
!pip install -q transformers accelerate peft evaluate jiwer datasets tensorboard soundfile librosa
print("✓ Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 80.5 MB/s eta 0:00:00:00:01
✓ Dependencies installed


In [ ]:
# Fix torchao version
!pip install -q --upgrade torchao
print("✓ torchao upgraded")

## 5. Verify GPU

In [6]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU detected! Runtime → Change runtime type → GPU")

CUDA available: True
GPU: Tesla T4
Memory: 15.6 GB


## 6. Run Training

In [10]:
# Set paths for training script
import os
os.environ['DATASET_PATH'] = str(DATA_ROOT)
os.environ['OUTPUT_DIR'] = str(OUTPUT_ROOT / 'whisper-fongbe')

!python scripts/finetune_whisper.py

🔧 FINETUNING WHISPER FONGBE
Modèle: openai/whisper-small
LoRA rank: 8
Target modules: ['q_proj', 'v_proj']

📦 Chargement dataset...
✅ Dataset chargé:
   Train: 10864 samples
   Validation: 1358 samples
   Test: 1359 samples

🤖 Chargement Whisper...
preprocessor_config.json: 100% 185k/185k [00:00<00:00, 84.0MB/s]
config.json: 100% 1.97k/1.97k [00:00<00:00, 5.09MB/s]
tokenizer_config.json: 100% 283k/283k [00:00<00:00, 280MB/s]
vocab.json: 100% 836k/836k [00:00<00:00, 50.1MB/s]
tokenizer.json: 100% 2.48M/2.48M [00:00<00:00, 142MB/s]
merges.txt: 100% 494k/494k [00:00<00:00, 98.7MB/s]
normalizer.json: 100% 52.7k/52.7k [00:00<00:00, 106MB/s]
added_tokens.json: 100% 34.6k/34.6k [00:00<00:00, 90.5MB/s]
special_tokens_map.json: 100% 2.19k/2.19k [00:00<00:00, 10.2MB/s]

model.safetensors: downloading bytes:  18% 171M/967M [00:01<00:04, 162MB/s, 15.6MB/s  ]  
model.safetensors: reconstructing file:  20% 194M/967M [00:01<00:06, 121MB/s]
model.safetensors: downloading bytes:  44% 430M/967M [00:02<0

## 7. TensorBoard (Optional)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_ROOT / 'whisper-fongbe'}

Launching TensorBoard...